# MiniMax-H3 + Pika-H3-Flex on one TPU v5e chip (Colab)

Text-to-video **with audio**, six steps, 480x832, 141 frames (about 6 s), on a single TPU v5e chip
(16 GB HBM, 47 GB host memory) — the free Colab TPU runtime.

> **Model terms.** MiniMax-H3 is released under the
> [MiniMax H3 Community License](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE), and the
> Pika-H3-Flex adapters are distributed under the same license. This notebook contains code only: it downloads
> the weights from the official Hugging Face repositories, and by doing so you accept their terms. Please read the
> license — including its territory restrictions and acceptable-use policy — and comply with it.
>
> **模型使用条款。** MiniMax-H3 以 MiniMax H3 社区许可证发布,Pika-H3-Flex 适配器沿用同一许可证。本 notebook
> 只包含代码,权重从官方 Hugging Face 仓库下载,下载即表示接受其条款。请阅读并遵守该许可证,包括其中的地域限制和可接受使用政策。

**Runtime:** *Runtime → Change runtime type → v5e-1 TPU*. Free-tier TPUs are not always available.

**Disk and time.** The first session downloads about 140 GB of weights and converts them to int8 (about 45 GB).
The free Colab TPU runtime has about 225 GB of disk, but a slow one: measured on 2026-11-05/06, the download took
about an hour and the first conversion several hours. Save the int8 cache to your own Cloud Storage bucket
(step 8) and later sessions restore it instead (step 3), which brings the setup down to well under an hour;
after that each video takes about a minute. Google Drive's free 15 GB is too small for the cache.
首次会话要下载约 140 GB 权重并转成 int8(约 45 GB)。免费 Colab TPU 运行时有约 225 GB 磁盘,但速度慢:2026-11-05/06 实测下载约一小时,首次转换要几个小时。把 int8 缓存存到你自己的 GCS bucket(第 8 步),之后的会话直接恢复(第 3 步),准备时间可缩短到一小时以内;之后每个视频约一分钟。Google Drive 免费的 15 GB 放不下这份缓存。

> **This copy has its outputs.** Executed end to end on a free Colab TPU v5e-1 runtime in November 2026, restoring the int8
> cache from a Cloud Storage bucket: cache restore ~10 min, download ~8 min, generation step 66 min
> (`generate: 81.5s`). Most of the generation step was the weight cache being read page by page from Colab's slow
> disk; this run predates commit `824a60d`, which reads it sequentially and should cut that to minutes. Run
> `h3_colab_v5e.ipynb` (no outputs) yourself.
>
> **这一份带输出。** 2026 年 11 月在免费 Colab TPU v5e-1 上从 GCS 恢复 int8 缓存后端到端跑完:恢复缓存约 10 分钟、
> 下载约 8 分钟、生成那一步 66 分钟(`generate: 81.5s`)。生成那一步大部分时间花在从 Colab 慢盘按页读取权重缓存上;这次运行早于
> 提交 `824a60d`,该提交改为顺序读取,预计能降到几分钟。自己运行请用不带输出的 `h3_colab_v5e.ipynb`。

In [1]:
# Settings
WORKDIR = "/content"
GCS_CACHE = "gs://<your-bucket>/h3-int8-cache"  #@param {type:"string"}
# ^ optional but recommended: a bucket path you can read and write, e.g. gs://<your-bucket>/h3-int8-cache (steps 3 and 8)
PROMPT = ("integrated_multimodal_description: A street musician plays a saxophone on a rainy city corner "
          "at night, neon signs reflecting on the wet pavement. The camera slowly circles him as he sways "
          "with the melody.\noverall_soundscape: Light rain on the pavement, distant traffic.\n"
          "non_diegetic_music: None. The saxophone plays a warm jazz melody, clearly audible over the rain.")
SEED = 42
NUM_FRAMES = 141
REPO = "https://github.com/shungcp/diffusers-tpu"
MINIMAX_REVISION = "42ed227ee7df40d41602854ae760620d6eb651fe"
PIKA_REVISION = "0ddf55ad7067ca1e76812253bd7b4600df87f759"
INSTALL = True

## 1. Check the runtime (seconds)

In [2]:
import os, shutil, subprocess
mem_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
disk_gb = shutil.disk_usage(WORKDIR).free / 2**30
print(f"host memory {mem_gb:.0f} GB, free disk {disk_gb:.0f} GB")
print(subprocess.run("ls /dev/vfio /dev/accel* 2>/dev/null | head -3", shell=True,
                     capture_output=True, text=True).stdout or "no TPU device found")
assert mem_gb > 40, "needs the v5e-1 TPU runtime (about 47 GB of host memory)"
if disk_gb < 200:
    print("WARNING: under 200 GB free; the download plus the int8 cache needs about 185 GB")

host memory 47 GB, free disk 225 GB
/dev/vfio:
0
vfio



## 2. Install (~5 min)

JAX 0.12 with libtpu, a CPU build of PyTorch (for torchax), and diffusers at a fixed commit.

In [3]:
%cd {WORKDIR}
if INSTALL:
    if not os.path.isdir("diffusers-tpu"):
        !git clone -q {REPO} diffusers-tpu
    !pip install -q -r diffusers-tpu/requirements/jax.txt && pip install -q -e diffusers-tpu
REPO_DIR = os.path.join(WORKDIR, "diffusers-tpu")

/content
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch-tpu 0.2.0 requires torch<=2.15.0,>=2.11.0, but you have torch 2.10.0+cpu which is incompatible.


## 3. Weight cache: restore from your bucket (seconds; ~20–30 min to restore 45 GB)

The first start converts the weights to int8 and writes them to `WEIGHT_CACHE` (about 45 GB). On free Colab
the disk is slow, so that first conversion takes hours; keep the cache in your own Cloud Storage bucket
(`GCS_CACHE` in the settings, saved by step 8) and later sessions restore it here instead.
Storing 45 GB costs roughly $1 a month, and reading it back may incur network charges.
首次启动会把权重转成 int8 写入 `WEIGHT_CACHE`(约 45 GB)。免费 Colab 的磁盘较慢,首次转换要几个小时;建议把缓存存进你自己的 GCS bucket(设置里的 `GCS_CACHE`,由第 8 步保存),之后的会话在这一步直接恢复。存 45 GB 大约每月 1 美元,读回时可能产生网络费用。

**Building the cache without a TPU (recommended).** The conversion runs on any CPU machine with about 100 GB
of memory and 200 GB of disk (for example a high-memory Cloud VM), and writes byte-for-byte the same files
as a TPU run. Use the same flags as step 6 so the cache matches:
**在没有 TPU 的机器上构建缓存(推荐)。** 转换可以在任意约 100 GB 内存、200 GB 磁盘的 CPU 机器上运行(比如一台高内存的云主机),生成的文件与 TPU 上逐字节相同。参数要和第 6 步一致,缓存才能对上:

```bash
git clone https://github.com/shungcp/diffusers-tpu && cd diffusers-tpu
pip install -r requirements/jax.txt && pip install -e .
PIKA=$(python -c "from huggingface_hub import snapshot_download as d; \
  d('MiniMaxAI/MiniMax-H3', revision='42ed227ee7df40d41602854ae760620d6eb651fe', max_workers=16, \
    allow_patterns=['transformer/*', 'text_encoder/*', 'tokenizer/*', 'vae/*', 'audio_vae/*', \
                    'scheduler/*', 'audio_scheduler/*', '*.json']); \
  print(d('Pika-Labs/Pika-H3-Flex', revision='0ddf55ad7067ca1e76812253bd7b4600df87f759', \
          allow_patterns=['fl2va/*', '*.json']))")
JAX_PLATFORMS=cpu DT_WEIGHT_CACHE=$PWD/h3_int8_cache PYTHON=$(which python) \
  bash scripts/run_h3_jax.sh t2va --chips 1 --int8-weights --adaln-host --cache-only \
  --pika-adapter $PIKA/fl2va --prompt x --out /tmp/h3_out
gcloud storage cp -r h3_int8_cache gs://<your-bucket>/      # then GCS_CACHE = "gs://<your-bucket>/h3_int8_cache"
```

In [6]:
WEIGHT_CACHE = os.environ.get("H3_WEIGHT_CACHE", os.path.join(WORKDIR, "h3_int8_cache"))
os.makedirs(WEIGHT_CACHE, exist_ok=True)
if GCS_CACHE:
    from google.colab import auth
    auth.authenticate_user()
    subprocess.run(["gcloud", "storage", "rsync", "-r", GCS_CACHE, WEIGHT_CACHE])
CACHE_COMPLETE = all(os.path.exists(os.path.join(WEIGHT_CACHE, d, "index.json"))
                     for d in ("text_int8", "dit_transformer_fl2va_int8_adaln6"))
print("cache complete" if CACHE_COMPLETE else "no complete cache: the first start builds it",
      subprocess.run(["du", "-sh", WEIGHT_CACHE], capture_output=True, text=True).stdout)

cache complete 44G	/content/h3_int8_cache



## 4. Download the weights (~40–60 min; ~5 min with a restored cache)
From the official repositories, pinned to the revisions this code was tested with: the text-to-video
transformer, the text encoder, both VAEs and the Pika text-to-video adapter.

On Colab this takes roughly 40–60 minutes, depending on the network. If the progress bar stops moving for
a long time, interrupt this cell and run it again: finished files are skipped and the download carries on.
在 Colab 上大约需要 40–60 分钟,视网络而定。进度条长时间不动时,中断这一格再重新运行即可续传,已下完的文件会跳过。

With a complete cache from step 3, the large text-encoder and transformer shards are skipped: only the
VAEs, configs and the adapter (~11 GB) are fetched.
步骤 3 恢复了完整缓存时,会跳过文本编码器和 transformer 的大分片,只下载 VAE、配置和适配器(约 11 GB)。

In [7]:
os.environ.setdefault("HF_HOME", os.path.join(WORKDIR, "hf"))
from huggingface_hub import snapshot_download
# The int8 cache replaces these shards; a cached start never reads them.
SKIP_SHARDS = (["text_encoder/*.safetensors", "transformer/*.safetensors"] if CACHE_COMPLETE else None)
PIKA = snapshot_download("Pika-Labs/Pika-H3-Flex", revision=PIKA_REVISION,
                         allow_patterns=["fl2va/*", "*.json", "LICENSE", "NOTICE"])
H3 = snapshot_download("MiniMaxAI/MiniMax-H3", revision=MINIMAX_REVISION, max_workers=16,
                       ignore_patterns=SKIP_SHARDS, allow_patterns=["transformer/*", "text_encoder/*", "tokenizer/*", "vae/*",
                                       "audio_vae/*", "scheduler/*", "audio_scheduler/*", "*.json", "LICENSE"])
print(H3, f"free disk now {shutil.disk_usage(WORKDIR).free / 2**30:.0f} GB")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 83 files:   0%|          | 0/83 [00:00<?, ?it/s]

/content/hf/hub/models--MiniMaxAI--MiniMax-H3/snapshots/42ed227ee7df40d41602854ae760620d6eb651fe free disk now 166 GB


## 5. Guard host memory (seconds)

1.   List item
2.   List item


47 GB is tight: the int8 weights stay in host memory and stream to the chip. Cap the page cache's dirty
pages so writing the weight cache cannot starve the process, and stop the run cleanly if memory runs out
instead of letting the whole runtime crash.

In [4]:
subprocess.run("sysctl -qw vm.dirty_bytes=1073741824 vm.dirty_background_bytes=268435456", shell=True)
GUARD = r'''
while true; do
  avail=$(awk '/MemAvailable/ {print int($2/1024)}' /proc/meminfo)
  pids=$(pgrep -f "diffusers_tpu[.]models" | tr '\n' ' ')
  if [ "${avail:-99999}" -lt 3072 ] && [ -n "$pids" ]; then
    echo "$(date +%T) memory guard: MemAvailable ${avail} MB, stopping $pids" >> /tmp/memguard.log
    kill -9 $pids
  fi
  sleep 2
done'''
guard = subprocess.Popen(["bash", "-c", GUARD])
print("memory guard running, pid", guard.pid)

memory guard running, pid 3159


## 6. Generate (first run: hours on free Colab; with a cache ~10 min; later runs ~2 min)

The first run converts the weights to int8 and writes the cache while loading (`text encoder loaded`,
then `transformer loaded`, about 6 minutes), then compiles and generates; later runs read the cache.
首次运行会在加载时把权重转成 int8 并写缓存(先后打印 `text encoder loaded`、`transformer loaded`,约 6 分钟),再编译和生成;之后的运行直接读缓存。

In [8]:
OUT = os.path.join(WORKDIR, "out")
import sys
env = dict(os.environ, PYTHON=sys.executable, DT_WEIGHT_CACHE=WEIGHT_CACHE, H3_COMPILE_CACHE=os.path.join(WORKDIR, "jax_cc"))
cmd = ["bash", "scripts/run_h3_jax.sh", "t2va", "--chips", "1", "--offload", "--int8-weights", "--adaln-host",
       "--height", "480", "--width", "832", "--num-frames", str(NUM_FRAMES), "--seed", str(SEED),
       "--pika-adapter", os.path.join(PIKA, "fl2va"), "--prompt", PROMPT, "--out", OUT, "--tag", "colab"]
import threading, time
def heartbeat(start=time.time()):
    # The first run prints nothing for a long while (reading and converting
    # the weights); show that it is alive.
    while proc.poll() is None:
        time.sleep(60)
        if proc.poll() is None:
            cache = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(WEIGHT_CACHE)
                        for f in fs) / 2**30
            print(f"  ... {int(time.time() - start) // 60} min, int8 cache {cache:.1f} GB, "
                  f"free disk {shutil.disk_usage(WORKDIR).free / 2**30:.0f} GB", flush=True)
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tail = []
threading.Thread(target=heartbeat, daemon=True).start()
for line in proc.stdout:
    tail = (tail + [line])[-30:]
    if any(k in line for k in ("generate", "loaded", "weight-cache", "denoise", "wrote")):
        print(line, end="")
if proc.wait() != 0:
    print("".join(tail))
    raise RuntimeError("generation failed; see /tmp/memguard.log if it was stopped for memory")
print(f"free disk now {shutil.disk_usage(WORKDIR).free / 2**30:.0f} GB")

  ... 1 min, int8 cache 43.8 GB, free disk 166 GB
  ... (2 heartbeat lines omitted)
  ... 4 min, int8 cache 43.8 GB, free disk 166 GB
  [weight-cache] text tower from /content/h3_int8_cache/text_int8
text encoder loaded  241s
  ... 5 min, int8 cache 43.8 GB, free disk 166 GB
  ... (25 heartbeat lines omitted)
  ... 31 min, int8 cache 43.8 GB, free disk 166 GB
  [t2va] text tower first 1593.4s (incl. compile; not part of generate)
  ... 32 min, int8 cache 43.8 GB, free disk 166 GB
  ... (3 heartbeat lines omitted)
  ... 36 min, int8 cache 43.8 GB, free disk 166 GB
  [weight-cache] DiT from /content/h3_int8_cache/dit_transformer_fl2va_int8_adaln6
transformer loaded  2180s
  ... 37 min, int8 cache 43.8 GB, free disk 166 GB
  ... (18 heartbeat lines omitted)
  ... 56 min, int8 cache 43.8 GB, free disk 166 GB
6 denoise calls  video shift 12.0 / audio shift 3.0
  ... 57 min, int8 cache 43.8 GB, free disk 166 GB
  ... (7 heartbeat lines omitted)
  ... 65 min, int8 cache 43.8 GB, free disk 166

## 7. Watch (seconds)

In [9]:
subprocess.run([sys.executable, os.path.join(REPO_DIR, "scripts/frames_to_mp4.py"), OUT,
                os.path.join(OUT, "colab_frames.npy")], check=True)
from IPython.display import Video
Video(os.path.join(OUT, "colab.mp4"), embed=True, width=832)

## 8. Keep the int8 cache in your bucket (recommended; ~30–60 min for 45 GB)

Set `SAVE_TO` to a bucket path you can write to (or set `GCS_CACHE` in the settings). The next session
restores it in step 3 and skips both the big download and the conversion.
把 `SAVE_TO` 设为你有写权限的 bucket 路径(或在设置里填 `GCS_CACHE`)。下次会话在第 3 步恢复它,既不用再下载大文件,也不用再转换。

In [ ]:
SAVE_TO = ""  #@param {type:"string"}
# ^ e.g. gs://<your-bucket>/h3-int8-cache; empty means the GCS_CACHE setting
SAVE_TO = SAVE_TO or GCS_CACHE
if SAVE_TO:
    from google.colab import auth
    auth.authenticate_user()
    subprocess.run(["gcloud", "storage", "rsync", "-r", WEIGHT_CACHE, SAVE_TO], check=True)
    print("saved to", SAVE_TO)
else:
    print("SAVE_TO is empty: the cache stays on this runtime and is lost when it is recycled")